# Compare `MyIEEE14_initialized` and Dynawo IEEE14 `IEEE14DisconnectLine`

This notebook mirrors `Compare_MyIEEE14_IEEE14NoEvent.ipynb`, but compares the generated initialized IEEE14 model against the Dynawo Modelica library example `Dynawo.Examples.IEEE14.TestCases.IEEE14DisconnectLine`.

The initialized package path is configurable in the first code cell. By default it uses the working copy created by the OpenModelica initialization workflow.


In [1]:
include("scripts/dictionaries.jl")
include("scripts/helpers.jl")
using .WorkflowHelpers

using OMJulia
using DataFrames
using Printf

# --- Configuration ---

# Local root for the OpenModelica-only notebooks
DEFAULT_ROOT_DIR = "/home/clarafercas/dynawo-notebooks/OpenModelica_only_users"
ROOT_DIR = isdir(joinpath(DEFAULT_ROOT_DIR, "Initialization")) ? DEFAULT_ROOT_DIR : pwd()

# Generated initialized package to validate.
# This defaults to the working copy produced by the initialization notebook.
# To compare another generated package instead, point this to its directory.
DEFAULT_AUX_PACKAGE_DIR = "/home/clarafercas/dynawo-notebooks/OpenModelica_only_users/Initialization/MyIEEE14_initialized"
BUILD_AUX_PACKAGE_DIR = joinpath(ROOT_DIR, "Initialization", "MyIEEE14_initialized")
AUX_PACKAGE_DIR = isdir(DEFAULT_AUX_PACKAGE_DIR) ? DEFAULT_AUX_PACKAGE_DIR : BUILD_AUX_PACKAGE_DIR
AUX_PACKAGE_FILE = joinpath(AUX_PACKAGE_DIR, "package.mo")
AUX_MODEL = "MyIEEE14_initialized.IEEE14DisconnectLine_initialized"

# Dynawo reference model
REFERENCE_PACKAGE_FILE = "/home/clarafercas/Model_library/dynawo/dynawo/sources/Models/Modelica/Dynawo/package.mo"
REFERENCE_MODEL = "Dynawo.Examples.IEEE14.TestCases.IEEE14DisconnectLine"

# Libraries
MODELICA_PKG_PATH = "/home/clarafercas/dynawo/OpenModelica/lib/omlibrary/Modelica/package.mo"
DYNAWO_PKG_PATH = REFERENCE_PACKAGE_FILE

# Comparison settings
STOP_TIME = 2000.0
MY_SLACK = "Gen1"
REFERENCE_SLACK = "Gen1"

println("Initialized package: ", AUX_PACKAGE_FILE)
println("Initialized model:   ", AUX_MODEL)
println("Reference model:     ", REFERENCE_MODEL)


Initialized package: /home/clarafercas/dynawo-notebooks/OpenModelica_only_users/Initialization/MyIEEE14_initialized/package.mo
Initialized model:   MyIEEE14_initialized.IEEE14DisconnectLine_initialized
Reference model:     Dynawo.Examples.IEEE14.TestCases.IEEE14DisconnectLine


In [2]:
const RESULT_FILE = IdDict()

function om_send(omc, expr; parsed = true)
    println("OMC -> ", expr)
    try
        return sendExpression(omc, expr; parsed = parsed)
    catch err
        println(sendExpression(omc, "getErrorString()", parsed = false))
        rethrow(err)
    end
end

function get_result_variable_names(omc, resultfile::String)
    vars = sendExpression(omc, "readSimulationResultVars(\"$resultfile\")")
    return sort!(String.(vars))
end

function collect_component_names_from_results(omc, resultfile::String, suffix::String; prefix = nothing, pattern = nothing)
    names = String[]
    seen = Set{String}()

    for var in get_result_variable_names(omc, resultfile)
        endswith(var, suffix) || continue
        name = chopsuffix(var, suffix)
        !isnothing(prefix) && !startswith(name, prefix) && continue
        !isnothing(pattern) && !occursin(pattern, name) && continue
        name in seen && continue
        push!(seen, name)
        push!(names, name)
    end

    sort!(names)
    return names
end

function read_last_value(sys, full_name::String)
    values = getSolutions(sys, full_name; resultfile = RESULT_FILE[sys])
    series = values[1]
    isempty(series) && error("No values found for $full_name")
    return Float64(series[end])
end

function read_complex(sys, base_name::String)
    re = read_last_value(sys, base_name * ".re")
    im = read_last_value(sys, base_name * ".im")
    return complex(re, im)
end

function voltage_mag_angle_deg_from_connector(sys, connector_path::String)
    v = read_complex(sys, connector_path * ".V")
    return abs(v), rad2deg(atan(imag(v), real(v)))
end

function terminal_power_pu(sys, component::String; terminal::String = "terminal")
    v = read_complex(sys, "$component.$terminal.V")
    i = read_complex(sys, "$component.$terminal.i")
    s = v * conj(i)
    return real(s), imag(s)
end

function run_and_simulate(label::String, package_file::String, model_name::String; stop_time::Float64 = STOP_TIME)
    isfile(package_file) || error("Package file not found: $package_file")

    omc = OMJulia.OMCSession()
    om_send(omc, "loadFile(\"$MODELICA_PKG_PATH\")")
    om_send(omc, "loadModel(Complex)")
    om_send(omc, "loadModel(ModelicaServices)")
    om_send(omc, "loadFile(\"$DYNAWO_PKG_PATH\")")
    package_file == DYNAWO_PKG_PATH || om_send(omc, "loadFile(\"$package_file\")")

    println("\nChecking $label...")
    chk = om_send(omc, "checkModel($model_name)", parsed = false)
    println(chk)

    build_dir = mktempdir()
    file_prefix = replace(lowercase(label), " " => "_")

    ModelicaSystem(
        omc,
        package_file,
        model_name,
        [MODELICA_PKG_PATH, DYNAWO_PKG_PATH],
        customBuildDirectory = build_dir,
    )

    n_intervals = round(Int, stop_time / 0.005)
    simflags = simulation_flags_without_log_stats(omc, model_name)
    om_send(omc, "simulate($model_name, startTime=0.0, stopTime=$(stop_time), numberOfIntervals=$(n_intervals), tolerance=1e-6, outputFormat=\"mat\", fileNamePrefix=\"$file_prefix\", simflags=\"$simflags\")", parsed = false)
    resultfile_path = joinpath(getWorkDirectory(omc), file_prefix * "_res.mat")
    RESULT_FILE[omc] = resultfile_path

    return Dict(
        "label" => label,
        "omc" => omc,
        "resultfile" => resultfile_path,
        "build_dir" => build_dir,
        "model" => model_name,
    )
end


run_and_simulate (generic function with 1 method)

In [3]:
function build_count_table(aux_counts::Dict{String,Int}, ref_counts::Dict{String,Int})
    component_types = ["buses", "non-slack generators", "transformers", "loads", "lines"]
    return DataFrame(
        component_type = component_types,
        auxiliary_count = [aux_counts[k] for k in component_types],
        dynawo_count = [ref_counts[k] for k in component_types],
        difference = [aux_counts[k] - ref_counts[k] for k in component_types],
    )
end

function build_voltage_table(aux_sys, ref_sys, components::Vector{String}; terminal::String = "terminal")
    rows = NamedTuple[]
    for component in sort(components)
        aux_u, aux_angle = voltage_mag_angle_deg_from_connector(aux_sys, "$component.$terminal")
        ref_u, ref_angle = voltage_mag_angle_deg_from_connector(ref_sys, "$component.$terminal")
        push!(rows, (
            component = component,
            auxiliary_u_pu = aux_u,
            dynawo_u_pu = ref_u,
            delta_u_pu = aux_u - ref_u,
            abs_delta_u_pu = abs(aux_u - ref_u),
            auxiliary_angle_deg = aux_angle,
            dynawo_angle_deg = ref_angle,
            delta_angle_deg = aux_angle - ref_angle,
            abs_delta_angle_deg = abs(aux_angle - ref_angle),
        ))
    end
    return DataFrame(rows)
end

function build_terminal_power_table(aux_sys, ref_sys, components::Vector{String}; terminal::String = "terminal")
    rows = NamedTuple[]
    for component in sort(components)
        aux_p, aux_q = terminal_power_pu(aux_sys, component; terminal = terminal)
        ref_p, ref_q = terminal_power_pu(ref_sys, component; terminal = terminal)
        push!(rows, (
            component = component,
            auxiliary_p_pu = aux_p,
            dynawo_p_pu = ref_p,
            delta_p_pu = aux_p - ref_p,
            abs_delta_p_pu = abs(aux_p - ref_p),
            auxiliary_q_pu = aux_q,
            dynawo_q_pu = ref_q,
            delta_q_pu = aux_q - ref_q,
            abs_delta_q_pu = abs(aux_q - ref_q),
        ))
    end
    return DataFrame(rows)
end

function max_abs_or_zero(df::DataFrame, col::Symbol)
    nrow(df) == 0 && return 0.0
    return maximum(df[!, col])
end

function build_summary_table(bus_voltage_df::DataFrame, generator_power_df::DataFrame, transformer_power_df::DataFrame, load_power_df::DataFrame, line_power_df::DataFrame, slack_power_df::DataFrame)
    return DataFrame(
        quantity = [
            "max |delta bus U| pu",
            "max |delta bus angle| deg",
            "max |delta generator P| pu",
            "max |delta generator Q| pu",
            "max |delta transformer terminal1 P| pu",
            "max |delta transformer terminal1 Q| pu",
            "max |delta load P| pu",
            "max |delta load Q| pu",
            "max |delta line terminal1 P| pu",
            "max |delta line terminal1 Q| pu",
            "|delta slack P| pu",
            "|delta slack Q| pu",
        ],
        value = [
            max_abs_or_zero(bus_voltage_df, :abs_delta_u_pu),
            max_abs_or_zero(bus_voltage_df, :abs_delta_angle_deg),
            max_abs_or_zero(generator_power_df, :abs_delta_p_pu),
            max_abs_or_zero(generator_power_df, :abs_delta_q_pu),
            max_abs_or_zero(transformer_power_df, :abs_delta_p_pu),
            max_abs_or_zero(transformer_power_df, :abs_delta_q_pu),
            max_abs_or_zero(load_power_df, :abs_delta_p_pu),
            max_abs_or_zero(load_power_df, :abs_delta_q_pu),
            max_abs_or_zero(line_power_df, :abs_delta_p_pu),
            max_abs_or_zero(line_power_df, :abs_delta_q_pu),
            max_abs_or_zero(slack_power_df, :abs_delta_p_pu),
            max_abs_or_zero(slack_power_df, :abs_delta_q_pu),
        ],
    )
end

function add_power_discrepancies!(rows::Vector{NamedTuple}, component_type::String, df::DataFrame)
    for row in eachrow(df)
        push!(rows, (
            component_type = component_type,
            component = row.component,
            score = max(row.abs_delta_p_pu, row.abs_delta_q_pu),
            delta_p_pu = row.delta_p_pu,
            delta_q_pu = row.delta_q_pu,
            delta_u_pu = missing,
            delta_angle_deg = missing,
        ))
    end
end

function add_voltage_discrepancies!(rows::Vector{NamedTuple}, component_type::String, df::DataFrame)
    for row in eachrow(df)
        push!(rows, (
            component_type = component_type,
            component = row.component,
            score = max(row.abs_delta_u_pu, row.abs_delta_angle_deg / 100),
            delta_p_pu = missing,
            delta_q_pu = missing,
            delta_u_pu = row.delta_u_pu,
            delta_angle_deg = row.delta_angle_deg,
        ))
    end
end

function build_top_discrepancy_table(bus_voltage_df::DataFrame, generator_power_df::DataFrame, transformer_power_df::DataFrame, load_power_df::DataFrame, line_power_df::DataFrame, slack_power_df::DataFrame; n::Int = 12)
    rows = NamedTuple[]
    add_voltage_discrepancies!(rows, "bus voltage", bus_voltage_df)
    add_power_discrepancies!(rows, "generator terminal power", generator_power_df)
    add_power_discrepancies!(rows, "transformer terminal1 power", transformer_power_df)
    add_power_discrepancies!(rows, "load terminal power", load_power_df)
    add_power_discrepancies!(rows, "line terminal1 power", line_power_df)
    add_power_discrepancies!(rows, "slack terminal power", slack_power_df)

    isempty(rows) && return DataFrame()
    df = DataFrame(rows)
    sort!(df, :score, rev = true)
    return first(df, min(n, nrow(df)))
end


build_top_discrepancy_table (generic function with 1 method)

In [4]:
aux_run = run_and_simulate("MyIEEE14 initialized", AUX_PACKAGE_FILE, AUX_MODEL; stop_time = STOP_TIME)
ref_run = run_and_simulate("Dynawo IEEE14DisconnectLine", REFERENCE_PACKAGE_FILE, REFERENCE_MODEL; stop_time = STOP_TIME)

aux_omc = aux_run["omc"]
ref_omc = ref_run["omc"]

println("\nInitialized result file: ", aux_run["resultfile"])
println("Reference result file: ", ref_run["resultfile"])


[ Info: Path to zmq file="/tmp/openmodelica.clarafercas.port.julia.PC2Yfz7jw5"


OMC -> loadFile("/home/clarafercas/dynawo/OpenModelica/lib/omlibrary/Modelica/package.mo")
OMC -> loadModel(Complex)
OMC -> loadModel(ModelicaServices)
OMC -> loadFile("/home/clarafercas/Model_library/dynawo/dynawo/sources/Models/Modelica/Dynawo/package.mo")
OMC -> loadFile("/home/clarafercas/dynawo-notebooks/OpenModelica_only_users/Initialization/MyIEEE14_initialized/package.mo")

Checking MyIEEE14 initialized...
OMC -> checkModel(MyIEEE14_initialized.IEEE14DisconnectLine_initialized)
"Check of MyIEEE14_initialized.IEEE14DisconnectLine_initialized completed successfully.
Class MyIEEE14_initialized.IEEE14DisconnectLine_initialized has 806 equation(s) and 806 variable(s).
308 of these are trivial equation(s)."

OMC -> simulate(MyIEEE14_initialized.IEEE14DisconnectLine_initialized, startTime=0.0, stopTime=2000.0, numberOfIntervals=400000, tolerance=1e-6, outputFormat="mat", fileNamePrefix="myieee14_initialized", simflags="-ls=klu -nls=kinsol -s=euler")


[ Info: Path to zmq file="/tmp/openmodelica.clarafercas.port.julia.D5qnH2Jb8l"


OMC -> loadFile("/home/clarafercas/dynawo/OpenModelica/lib/omlibrary/Modelica/package.mo")
OMC -> loadModel(Complex)
OMC -> loadModel(ModelicaServices)
OMC -> loadFile("/home/clarafercas/Model_library/dynawo/dynawo/sources/Models/Modelica/Dynawo/package.mo")

Checking Dynawo IEEE14DisconnectLine...
OMC -> checkModel(Dynawo.Examples.IEEE14.TestCases.IEEE14DisconnectLine)
"Check of Dynawo.Examples.IEEE14.TestCases.IEEE14DisconnectLine completed successfully.
Class Dynawo.Examples.IEEE14.TestCases.IEEE14DisconnectLine has 784 equation(s) and 784 variable(s).
286 of these are trivial equation(s)."

OMC -> simulate(Dynawo.Examples.IEEE14.TestCases.IEEE14DisconnectLine, startTime=0.0, stopTime=2000.0, numberOfIntervals=400000, tolerance=1e-6, outputFormat="mat", fileNamePrefix="dynawo_ieee14disconnectline", simflags="-ls=klu -nls=kinsol -s=euler")

Initialized result file: /tmp/jl_Waa3ew/myieee14_initialized_res.mat
Reference result file: /tmp/jl_tKOpvq/dynawo_ieee14disconnectline_res.mat


In [5]:
aux_buses = collect_component_names_from_results(aux_omc, aux_run["resultfile"], ".terminal.V.re"; prefix = "Bus")
ref_buses = collect_component_names_from_results(ref_omc, ref_run["resultfile"], ".terminal.V.re"; prefix = "Bus")
common_buses = sort(intersect(aux_buses, ref_buses))

aux_generators = setdiff(collect_component_names_from_results(aux_omc, aux_run["resultfile"], ".terminal.V.re"; prefix = "Gen"), [MY_SLACK])
ref_generators = setdiff(collect_component_names_from_results(ref_omc, ref_run["resultfile"], ".terminal.V.re"; prefix = "Gen"), [REFERENCE_SLACK])
common_generators = sort(intersect(aux_generators, ref_generators))

aux_transformers = collect_component_names_from_results(aux_omc, aux_run["resultfile"], ".terminal1.V.re"; prefix = "Tfo")
ref_transformers = collect_component_names_from_results(ref_omc, ref_run["resultfile"], ".terminal1.V.re"; prefix = "Tfo")
common_transformers = sort(intersect(aux_transformers, ref_transformers))

aux_loads = collect_component_names_from_results(aux_omc, aux_run["resultfile"], ".terminal.V.re"; prefix = "Load")
ref_loads = collect_component_names_from_results(ref_omc, ref_run["resultfile"], ".terminal.V.re"; prefix = "Load")
common_loads = sort(intersect(aux_loads, ref_loads))

aux_lines = collect_component_names_from_results(aux_omc, aux_run["resultfile"], ".terminal1.V.re"; prefix = "Line")
ref_lines = collect_component_names_from_results(ref_omc, ref_run["resultfile"], ".terminal1.V.re"; prefix = "Line")
common_lines = sort(intersect(aux_lines, ref_lines))

count_comparison_df = build_count_table(
    Dict(
        "buses" => length(aux_buses),
        "non-slack generators" => length(aux_generators),
        "transformers" => length(aux_transformers),
        "loads" => length(aux_loads),
        "lines" => length(aux_lines),
    ),
    Dict(
        "buses" => length(ref_buses),
        "non-slack generators" => length(ref_generators),
        "transformers" => length(ref_transformers),
        "loads" => length(ref_loads),
        "lines" => length(ref_lines),
    ),
)

bus_voltage_df = build_voltage_table(aux_omc, ref_omc, common_buses)
generator_power_df = build_terminal_power_table(aux_omc, ref_omc, common_generators)
transformer_power_df = build_terminal_power_table(aux_omc, ref_omc, common_transformers; terminal = "terminal1")
load_power_df = build_terminal_power_table(aux_omc, ref_omc, common_loads)
line_power_df = build_terminal_power_table(aux_omc, ref_omc, common_lines; terminal = "terminal1")
slack_power_df = build_terminal_power_table(aux_omc, ref_omc, [MY_SLACK])

summary_df = build_summary_table(bus_voltage_df, generator_power_df, transformer_power_df, load_power_df, line_power_df, slack_power_df)
top_discrepancy_df = build_top_discrepancy_table(bus_voltage_df, generator_power_df, transformer_power_df, load_power_df, line_power_df, slack_power_df)


Row,component_type,component,score,delta_p_pu,delta_q_pu,delta_u_pu,delta_angle_deg
,String,String,Float64,Float64?,Float64?,Float64?,Float64?
1,line terminal1 power,LineB1B2,1.48879e-5,1.48879e-5,-2.76801e-6,missing,missing
2,slack terminal power,Gen1,1.48879e-5,-1.48879e-5,2.76801e-6,missing,missing
3,generator terminal power,Gen2,1.36407e-5,1.36407e-5,-6.57602e-6,missing,missing
4,bus voltage,Bus2,4.87656e-7,missing,missing,0.0,-4.87656e-5
5,bus voltage,Bus3,4.87656e-7,missing,missing,2.22045e-16,-4.87656e-5
6,bus voltage,Bus5,4.87656e-7,missing,missing,9.10383e-15,-4.87656e-5
7,bus voltage,Bus4,4.87656e-7,missing,missing,8.88178e-15,-4.87656e-5
8,bus voltage,Bus7,4.87656e-7,missing,missing,6.21725e-15,-4.87656e-5
9,bus voltage,Bus8,4.87656e-7,missing,missing,2.22045e-16,-4.87656e-5


In [6]:
println("Counts in both models:")
display(count_comparison_df)

println("Max absolute differences:")
display(summary_df)

println("Top discrepancies across buses, generators, transformers, loads, lines, and slack:")
display(top_discrepancy_df)

println("Non-slack generator terminal P/Q:")
display(generator_power_df[:, [:component, :auxiliary_p_pu, :dynawo_p_pu, :delta_p_pu, :auxiliary_q_pu, :dynawo_q_pu, :delta_q_pu]])

println("Slack terminal P/Q:")
display(slack_power_df[:, [:component, :auxiliary_p_pu, :dynawo_p_pu, :delta_p_pu, :auxiliary_q_pu, :dynawo_q_pu, :delta_q_pu]])

println("Bus voltage magnitude and angle:")
display(bus_voltage_df[:, [:component, :auxiliary_u_pu, :dynawo_u_pu, :delta_u_pu, :auxiliary_angle_deg, :dynawo_angle_deg, :delta_angle_deg]])

println("Load terminal P/Q:")
display(load_power_df[:, [:component, :auxiliary_p_pu, :dynawo_p_pu, :delta_p_pu, :auxiliary_q_pu, :dynawo_q_pu, :delta_q_pu]])


Counts in both models:


Row,component_type,auxiliary_count,dynawo_count,difference
,String,Int64,Int64,Int64
1,buses,14,14,0
2,non-slack generators,4,4,0
3,transformers,3,3,0
4,loads,11,11,0
5,lines,17,17,0


Max absolute differences:


Row,quantity,value
,String,Float64
1,max |delta bus U| pu,1.15463e-14
2,max |delta bus angle| deg,4.87656e-5
3,max |delta generator P| pu,1.36407e-5
4,max |delta generator Q| pu,6.57602e-6
5,max |delta transformer terminal1 P| pu,3.33067e-14
6,max |delta transformer terminal1 Q| pu,3.86635e-14
7,max |delta load P| pu,3.51164e-13
8,max |delta load Q| pu,4.77535e-14
9,max |delta line terminal1 P| pu,1.48879e-5


Top discrepancies across buses, generators, transformers, loads, lines, and slack:


...

Row,component,auxiliary_p_pu,dynawo_p_pu,delta_p_pu,auxiliary_q_pu,dynawo_q_pu,delta_q_pu
,String,Float64,Float64,Float64,Float64,Float64,Float64
1,Gen1,-2.36486,-2.36484,-1.48879e-5,0.37221,0.372208,2.76801e-6


Bus voltage magnitude and angle:


Row,component,auxiliary_u_pu,dynawo_u_pu,delta_u_pu,auxiliary_angle_deg,dynawo_angle_deg,delta_angle_deg
,String,Float64,Float64,Float64,Float64,Float64,Float64
1,Bus1,1.06,1.06,0.0,-7.3499e-26,-2.62056e-23,2.61321e-23
2,Bus10,1.04847,1.04847,8.65974e-15,-20.7819,-20.7818,-4.87656e-5
3,Bus11,1.05581,1.05581,4.66294e-15,-20.6536,-20.6536,-4.87656e-5
4,Bus12,1.05509,1.05509,1.11022e-15,-21.0993,-21.0992,-4.87656e-5
5,Bus13,1.05022,1.05022,2.66454e-15,-21.1436,-21.1436,-4.87656e-5
6,Bus14,1.03365,1.03365,1.15463e-14,-21.8092,-21.8092,-4.87656e-5
7,Bus2,1.04507,1.04507,0.0,-7.60295,-7.6029,-4.87656e-5
8,Bus3,1.01,1.01,2.22045e-16,-16.8552,-16.8551,-4.87656e-5
9,Bus4,1.00865,1.00865,8.88178e-15,-15.6454,-15.6453,-4.87656e-5


Load terminal P/Q:


Row,component,auxiliary_p_pu,dynawo_p_pu,delta_p_pu,auxiliary_q_pu,dynawo_q_pu,delta_q_pu
,String,Float64,Float64,Float64,Float64,Float64,Float64
1,Load10,0.09,0.09,-2.28151e-14,0.058,0.058,-2.45012e-14
2,Load11,0.0352907,0.0352907,2.22045e-16,0.0182499,0.0182499,1.94289e-16
3,Load12,0.0614441,0.0614441,9.02056e-17,0.0161946,0.0161946,5.20417e-17
4,Load13,0.135042,0.135042,5.27356e-16,0.0580303,0.0580303,3.60822e-16
5,Load14,0.149,0.149,-3.64431e-14,0.05,0.05,-2.03657e-14
6,Load2,0.217,0.217,-2.77556e-17,0.127,0.127,0.0
7,Load3,0.942,0.942,-1.11022e-16,0.19,0.19,1.11022e-16
8,Load4,0.478,0.478,-3.51164e-13,-0.039,-0.039,4.77535e-14
9,Load5,0.076,0.076,-7.80209e-14,0.016,0.016,-2.73913e-14


## Notes

This comparison uses the initialized user-style model against the Dynawo library example with the same disconnect-line event. Small numerical differences can still appear because the two simulations may not emit exactly the same output grid or may follow slightly different internal initialization paths, even when the electrical trajectories match closely.
